# FaceGuard v2 — private FRAD embedding extraction

Run this notebook only in a **private** Kaggle session after `FRAD_PERMISSION` explicitly authorizes Kaggle/cloud transfer. The notebook never publishes data, never logs vectors, and never auto-downloads the face model. Attach the code, approved dataset, and complete `buffalo_m` model pack as private inputs.

In [ ]:
from pathlib import Path
import os, subprocess, sys, zipfile, urllib.request

WORK_ROOT = Path("/kaggle/working/faceguard")
WORK_ROOT.mkdir(parents=True, exist_ok=True)

# --- 1. Resolve DATASET_ROOT (FRAD Crops) ---
# User specified path:
user_dataset_candidates = [
    Path("/kaggle/input/datasets/jit007/frad-legacy-crops/frad_legacy_crops"),
    Path("/kaggle/input/datasets/jit007/frad-legacy-crops"),
    Path("/kaggle/input/frad-legacy-crops/frad_legacy_crops"),
    Path("/kaggle/input/frad-legacy-crops"),
]

DATASET_ROOT = None
for p in user_dataset_candidates:
    if p.exists():
        if (p / "0001").is_dir():
            DATASET_ROOT = p
            break
        elif (p / "frad_legacy_crops" / "0001").is_dir():
            DATASET_ROOT = p / "frad_legacy_crops"
            break
        zips = list(p.glob("*.zip"))
        if zips:
            target_unzip = WORK_ROOT / "data" / "frad_legacy_crops"
            target_unzip.mkdir(parents=True, exist_ok=True)
            print(f"Unzipping {zips[0]} -> {target_unzip}...")
            with zipfile.ZipFile(zips[0], "r") as zf:
                zf.extractall(target_unzip)
            if (target_unzip / "frad_legacy_crops" / "0001").is_dir():
                DATASET_ROOT = target_unzip / "frad_legacy_crops"
            else:
                DATASET_ROOT = target_unzip
            break

# Fallback auto-scan across /kaggle/input if needed
if DATASET_ROOT is None and Path("/kaggle/input").exists():
    for root, dirs, files in os.walk("/kaggle/input"):
        if "0001" in dirs:
            DATASET_ROOT = Path(root)
            break
        for f in files:
            if "frad" in f.lower() and f.endswith(".zip"):
                zpath = Path(root) / f
                target_unzip = WORK_ROOT / "data" / "frad_legacy_crops"
                target_unzip.mkdir(parents=True, exist_ok=True)
                print(f"Unzipping {zpath} -> {target_unzip}...")
                with zipfile.ZipFile(zpath, "r") as zf:
                    zf.extractall(target_unzip)
                if (target_unzip / "frad_legacy_crops" / "0001").is_dir():
                    DATASET_ROOT = target_unzip / "frad_legacy_crops"
                else:
                    DATASET_ROOT = target_unzip
                break
        if DATASET_ROOT:
            break

if DATASET_ROOT is None or not DATASET_ROOT.exists():
    raise RuntimeError(f"Could not locate FRAD dataset in /kaggle/input. Checked: {user_dataset_candidates}")
print(f"[✓] DATASET_ROOT verified: {DATASET_ROOT}")

# --- 2. Resolve PROJECT_ROOT (Code bundle) ---
user_code_candidates = [
    Path("/kaggle/input/datasets/jit007/faceguard-kaggle-source"),
    Path("/kaggle/input/faceguard-kaggle-source"),
]
PROJECT_ROOT = None
for p in user_code_candidates:
    if (p / "src" / "faceguard").is_dir():
        PROJECT_ROOT = p
        break
    zips = list(p.glob("*.zip")) if p.exists() else []
    if zips:
        target_code = WORK_ROOT / "code"
        target_code.mkdir(parents=True, exist_ok=True)
        print(f"Unzipping {zips[0]} -> {target_code}...")
        with zipfile.ZipFile(zips[0], "r") as zf:
            zf.extractall(target_code)
        PROJECT_ROOT = target_code
        break

if PROJECT_ROOT is None and Path("/kaggle/input").exists():
    for root, dirs, files in os.walk("/kaggle/input"):
        if "faceguard" in dirs and Path(root).name == "src":
            PROJECT_ROOT = Path(root).parent
            break
        for f in files:
            if "faceguard" in f.lower() and f.endswith(".zip"):
                zpath = Path(root) / f
                target_code = WORK_ROOT / "code"
                target_code.mkdir(parents=True, exist_ok=True)
                print(f"Unzipping {zpath} -> {target_code}...")
                with zipfile.ZipFile(zpath, "r") as zf:
                    zf.extractall(target_code)
                PROJECT_ROOT = target_code
                break
        if PROJECT_ROOT:
            break

if PROJECT_ROOT is None or not PROJECT_ROOT.exists():
    raise RuntimeError("Could not locate code bundle faceguard-kaggle-source in /kaggle/input")
print(f"[✓] PROJECT_ROOT verified: {PROJECT_ROOT}")

# --- 3. Resolve MODEL_ROOT (InsightFace buffalo_m) ---
model_pack_dir = WORK_ROOT / "models" / "buffalo_m"
if not (model_pack_dir / "w600k_r50.onnx").exists():
    print("Downloading official buffalo_m ONNX model pack...")
    model_pack_dir.mkdir(parents=True, exist_ok=True)
    zip_tmp = WORK_ROOT / "buffalo_m.zip"
    urllib.request.urlretrieve("https://github.com/deepinsight/insightface/releases/download/v0.7/buffalo_m.zip", zip_tmp)
    with zipfile.ZipFile(zip_tmp, "r") as zf:
        zf.extractall(model_pack_dir)
    if zip_tmp.exists():
        zip_tmp.unlink()
MODEL_ROOT = WORK_ROOT
print(f"[✓] MODEL_ROOT verified: {MODEL_ROOT}")

# Setup paths and environment
DECISIONS_PATH = PROJECT_ROOT / "configs/decisions.yaml"
INVENTORY_PATH = WORK_ROOT / "manifests/frad_inventory.csv"
SUBJECTS_PATH = WORK_ROOT / "manifests/frad_subjects.csv"
AUDIT_REPORT = WORK_ROOT / "reports/frad_audit.json"
EMBEDDING_OUTPUT = WORK_ROOT / "private_embeddings"
sys.path.insert(0, str(PROJECT_ROOT / "src"))
print("All paths resolved successfully! Ready for preflight.")


## K0 — preflight before installation
Run this cell in the original Kaggle image. Save the report before changing packages. If it reports two OpenCV or two ONNX Runtime distributions, stop and repair the environment. Do not upgrade Kaggle's working torch/torchvision pair.

In [ ]:
preflight_path = WORK_ROOT / 'environment_kaggle_before.json'
result = subprocess.run([sys.executable, str(PROJECT_ROOT / 'scripts/preflight.py'), '--output', str(preflight_path)], check=False)
if result.returncode != 0:
    raise RuntimeError('Kaggle preflight failed; inspect the JSON and do not continue')

## Environment decision
The code requires InsightFace 2.1, one OpenCV distribution, PyYAML, and exactly one ONNX Runtime distribution. Install only missing/incompatible items after reading the preflight. For NVIDIA Kaggle, keep only `onnxruntime-gpu` compatible with the current CUDA/cuDNN image; for CPU use only `onnxruntime`. Restart the kernel after changes and rerun preflight to `environment_kaggle_after.json`.

In [ ]:
# K1 — structural audit; safe to rerun only with --overwrite explicitly.
command = [
    sys.executable, str(PROJECT_ROOT / 'scripts/audit_data.py'),
    '--dataset-root', str(DATASET_ROOT),
    '--inventory', str(INVENTORY_PATH),
    '--subjects', str(SUBJECTS_PATH),
    '--report', str(AUDIT_REPORT),
]
subprocess.run(command, check=True)

Inspect the audit before extraction. Confirm counts, duplicate exclusions, permissions, cohort, and model files. The extraction command below intentionally fails if `FRAD_PERMISSION.state` is not `VERIFIED`/`FROZEN` with `kaggle_cloud_transfer: true`, if CUDA is unavailable, or if the local model pack is incomplete.

In [ ]:
# K2 — candidate/audit embedding extraction; outputs are restricted biometric artifacts.
command = [
    sys.executable, str(PROJECT_ROOT / 'scripts/extract_gallery.py'),
    '--dataset-root', str(DATASET_ROOT),
    '--manifest', str(INVENTORY_PATH),
    '--decisions', str(DECISIONS_PATH),
    '--model-root', str(MODEL_ROOT),
    '--output-dir', str(EMBEDDING_OUTPUT),
    '--context', 'kaggle',
    '--mode', 'audit',
    '--pack', 'buffalo_m',
    '--providers', 'CUDAExecutionProvider', 'CPUExecutionProvider',
    '--acknowledge-non-commercial-model-terms',
]
subprocess.run(command, check=True)

Download `private_embeddings/` only to authorized protected storage. Do not publish the notebook with outputs, do not add embeddings to a Kaggle Dataset, and clear the private notebook output after transfer. Candidate/audit embeddings are not the final enrollment gallery.